# Confronto esplorativo: WMH vs ATLAS

Questo notebook ispeziona i dati condivisi in sola lettura. Si concentra su risoluzione spaziale, geometria, allineamento immagine-label, visualizzazione e conseguenze per resampling/padding. Non modifica né copia i volumi.

Le chiavi W&B non vengono lette né mostrate in questo notebook.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

ROOT = Path('/home/ccantini/lesion_detection_project')
CFG = json.loads((ROOT / 'configs' / 'datasets.json').read_text())
WMH_ROOT = Path(CFG['datasets']['WMH']['path'])
ATLAS_ROOT = Path(CFG['datasets']['ATLAS']['path'])
print('WMH:', WMH_ROOT)
print('ATLAS:', ATLAS_ROOT)


In [ ]:
def wmh_cases():
    cases = []
    for flair in sorted(WMH_ROOT.rglob('*_FLAIR_BiasCorr.nii.gz')):
        label = flair.with_name(flair.name.replace('_FLAIR_BiasCorr.nii.gz', '_wmh.nii.gz'))
        t1 = flair.with_name(flair.name.replace('_FLAIR_BiasCorr.nii.gz', '_T1w_BiasCorr_2FLAIR_Aligned.nii.gz'))
        if label.exists():
            cases.append({'id': flair.parent.name, 'image': flair, 'label': label, 't1': t1 if t1.exists() else None})
    return cases

def atlas_cases(split='Training'):
    cases = []
    for image in sorted((ATLAS_ROOT / split).rglob('*_T1w.nii.gz')):
        label = image.with_name(image.name.replace('_T1w.nii.gz', '_label-L_desc-T1lesion_mask.nii.gz'))
        cases.append({'id': image.name.split('_ses-')[0], 'image': image, 'label': label if label.exists() else None, 't1': None})
    return cases

wmh = wmh_cases(); atlas_tr = atlas_cases('Training'); atlas_ts = atlas_cases('Testing')
print(f'WMH paired cases: {len(wmh)}')
print(f'ATLAS train paired cases: {sum(x["label"] is not None for x in atlas_tr)}/{len(atlas_tr)}')
print(f'ATLAS test images: {len(atlas_ts)} (labels intentionally absent)')


In [ ]:
def metadata(case, dataset):
    image = nib.load(str(case['image']), mmap=True)
    label_ok = None
    if case['label'] is not None:
        label = nib.load(str(case['label']), mmap=True)
        label_ok = image.shape[:3] == label.shape[:3] and np.allclose(image.affine, label.affine, atol=1e-3)
    zoom = image.header.get_zooms()[:3]
    return {'dataset': dataset, 'id': case['id'], 'shape': tuple(image.shape[:3]), 'sx_mm': zoom[0], 'sy_mm': zoom[1], 'sz_mm': zoom[2], 'fov_x_mm': image.shape[0]*zoom[0], 'fov_y_mm': image.shape[1]*zoom[1], 'fov_z_mm': image.shape[2]*zoom[2], 'image_label_aligned': label_ok}

df = pd.DataFrame([metadata(c, 'WMH') for c in wmh] + [metadata(c, 'ATLAS train') for c in atlas_tr] + [metadata(c, 'ATLAS test') for c in atlas_ts])
display(df.groupby('dataset')[[ 'sx_mm','sy_mm','sz_mm','fov_x_mm','fov_y_mm','fov_z_mm']].agg(['count','median','min','max']).round(3))
display(df.groupby('dataset')['image_label_aligned'].value_counts(dropna=False))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for dataset, color in [('WMH', 'tab:blue'), ('ATLAS train', 'tab:orange')]:
    part = df[df.dataset == dataset]
    for col, ax in zip(['sx_mm','sy_mm','sz_mm'], axes):
        ax.hist(part[col], bins=25, alpha=.55, label=dataset, color=color)
        ax.set_title(col); ax.set_xlabel('mm'); ax.legend()
plt.suptitle('Distribuzione della risoluzione spaziale'); plt.tight_layout()


In [ ]:
def show_case(case, title):
    image = nib.load(str(case['image'])); data = image.get_fdata(dtype=np.float32)
    z = data.shape[2] // 2
    fig, ax = plt.subplots(1, 2 if case['label'] is not None else 1, figsize=(10, 5))
    ax = np.atleast_1d(ax)
    ax[0].imshow(np.rot90(data[:, :, z]), cmap='gray', vmin=np.percentile(data, 1), vmax=np.percentile(data, 99))
    ax[0].set_title(f'{title}: image, z={z}'); ax[0].axis('off')
    if case['label'] is not None:
        lab = nib.load(str(case['label'])).get_fdata()
        ax[1].imshow(np.rot90(data[:, :, z]), cmap='gray', vmin=np.percentile(data, 1), vmax=np.percentile(data, 99))
        ax[1].imshow(np.rot90(lab[:, :, z]), cmap='autumn', alpha=.55, vmin=0, vmax=max(1, lab.max()))
        ax[1].set_title('Overlay label'); ax[1].axis('off')
    plt.show()

show_case(wmh[0], 'WMH FLAIR')
show_case(atlas_tr[0], 'ATLAS T1w')


## Interpretazione per nnU-Net

- **WMH**: le voxel size sono eterogenee e spesso anisotrope. FLAIR è la baseline più informativa; T1 allineato può essere un secondo canale. Il planner nnU-Net dovrebbe scegliere la spacing target: non forzare subito 1 mm isotropico, perché aumenta molto i voxel lungo z.
- **ATLAS**: T1w in spazio MNI con spacing vicino a 1 mm isotropico; partire T1-only. Selezionare una sola variante di maschera come target.
- **Padding**: utile per rendere compatibili le dimensioni con il patch size dopo resampling. Deve essere applicato alla fine del preprocessing, con valore 0 per l'immagine e background per la label. nnU-Net lo gestisce internamente durante preprocessing/inferenza; non serve fare padding manuale ai raw data salvo esperimenti specifici.
- **Resampling**: interpolazione continua per immagini, nearest-neighbour per label. Controllare sempre overlay e affini dopo la conversione.
- **Split**: mai inserire test ufficiali nel training; per WMH mantenere, quando possibile, una valutazione per sito/scanner.